## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

In [2]:
# Model 
from langchain_openai import ChatOpenAI

model = ChatOpenAI()
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001F483775D30>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F4837767B0>, root_client=<openai.OpenAI object at 0x000001F48352EE40>, root_async_client=<openai.AsyncOpenAI object at 0x000001F483776510>, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import AIMessage, HumanMessage

model.invoke(
    [
        HumanMessage(content="Hi, my name is Hemant and i am Agentic ai Engginer"),
        AIMessage(content="Nice to meet you Hemant! It's great to have someone with expertise in Agentic AI on our team. How did you become interested in this field?"),
        HumanMessage(content="Hey, what is my name and what i do?")
    ]
)

AIMessage(content="Your name is Hemant and you are an Agentic AI Engineer. Let me know if there's anything else you want to know or talk about!", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 74, 'total_tokens': 104, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CBcP7I0xsP5Ft7Mr1P5fNKt8WmUK3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--5c42b241-bea5-4313-bc02-6ef183ffa9f0-0', usage_metadata={'input_tokens': 74, 'output_tokens': 30, 'total_tokens': 104, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [5]:
config = {'configurable' : {'session_id' : 'chat1'}}

In [6]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is Hemant and i am Agentic ai Engginer")
       
    ],
    config=config
)

response.content

"Nice to meet you, Hemant! It's great to hear that you are an Agentic AI Engineer. What kind of projects do you work on in your role?"

In [7]:
with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name")
    ],
    config=config
)

AIMessage(content='Your name is Hemant.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 71, 'total_tokens': 77, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CBcPE1Tr9e1Frdiwqh8XhSPMxH8KI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--b15de619-2d05-4c1a-ba9e-265da4e318b3-0', usage_metadata={'input_tokens': 71, 'output_tokens': 6, 'total_tokens': 77, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [8]:
# Same session id
config1 = {'configurable': {'session_id': 'chat1'}}

response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name")
    ],
    config=config1
)
response.content



'Your name is Hemant.'

In [9]:
# different session id
config2 = {'configurable': {'session_id': 'chat2'}}

response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name")
    ],
    config=config2
)
response.content

"I'm sorry, I cannot know your name as I am an AI assistant and do not have the capability to remember personal details about individuals."

In [10]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is rahul")
    ],
    config=config2
)
response.content

'Hello Rahul! How can I assist you today?'

In [11]:
response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, What is my name")
    ],
    config=config2
)
response.content

'Your name is Rahul.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "you are a helpful assistant"),
        MessagesPlaceholder(variable_name="messages")
        
    ]
)

# Chain
chain = prompt | model



In [13]:
chain.invoke({'messages': [HumanMessage(content="Hi, my name is sp")]})

AIMessage(content='Hello sp! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 22, 'total_tokens': 32, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CBcPNdB13DUMfxFY57EFeMKzKbRoO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--ae01e3c1-87f8-4e63-93c0-1f5ee275fa5f-0', usage_metadata={'input_tokens': 22, 'output_tokens': 10, 'total_tokens': 32, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [14]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)



In [15]:
config3 = {'configurable': {'session_id': 'chat3'}}

response = with_message_history.invoke(
    [
        HumanMessage(content="Hi, my name is sp")
    ],
    config=config3
)

response.content

'Hello! Nice to meet you, sp. How can I assist you today?'

In [16]:
# add more complexcity

# Prompt
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "you are a helpful assistant language: {language}"),
        MessagesPlaceholder(variable_name="messages")
        
    ]
)

# Chain
chain = prompt | model

In [17]:
response = chain.invoke({'messages': [HumanMessage(content="Hi, my name is sp")], "language": "Hindi"})

response.content

'नमस्ते, sp! मैं आपकी सहायक हूँ। कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [18]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='messages'
)

In [19]:
config4 = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config4
)
repsonse.content

'नमस्ते, कृष्ण। आपकी सहायक कैसे हो सकती हूँ?'

In [20]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config4,
)
response.content

'आपका नाम कृष्ण है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

## trim_messages
LangChain comes with a few built-in helpers for managing a list of messages. In this case we'll use the trim_messages helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages:

In [21]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [22]:
from operator import  itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

response = chain.invoke(
    {
        'messages': messages + [HumanMessage(content="what ice cream do i like?")],
        "language":"English"
    }
)

response.content

"I'm not sure, what's your favorite ice cream flavor?"

In [23]:
response = chain.invoke(
    {
        'messages': messages + [HumanMessage(content="what math problem did i ask?")],
        "language":"English"
    }
)

response.content

'You asked "what\'s 2 + 2?"'

In [24]:
## Lets wrap this in the Message History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='messages'
)
config5 = {"configurable":{"session_id":"chat5"}}

In [25]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config5,
)

response.content

"I'm sorry, I don't have access to your personal information."

In [26]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config5,
)

response.content

'you did not ask a math problem yet'